# Lab 12 — Hadoop: FileSystem, MapReduce e YARN

> Extraído de `aulas/Aula_9_hadoop.ipynb` (seções 1 a 5 e 7, exercícios 1, 2 e 4; não exige instalar o Hadoop. A seção 6, o exercício 3 e o resumo estão no notebook 02). As referências a "aula N" e a números de seção usam a numeração do notebook original.

# Aula 9 — Hadoop: o que está por baixo do Spark
### HDFS, MapReduce, YARN e como tudo se conecta

Em todas as aulas usamos caminhos como `file:///tmp/...`, configurações como `spark.hadoop.*` e o Spark falou de "blocos" e "splits". Tudo isso vem do **Hadoop**, o ecossistema sobre o qual o Spark foi construído.

Nesta aula de fechamento vamos abrir essa camada: o que o Hadoop resolveu, por que o Spark o substituiu em parte, o que continua sendo usado hoje e como Spark e Pandas conversam com ele.

**Objetivos**
- Entender os três pilares do Hadoop: HDFS, MapReduce e YARN
- Implementar MapReduce em Python, no estilo Hadoop Streaming, e comparar com Spark
- Ver como o Spark usa a API de FileSystem do Hadoop (e por que `s3a://` funciona)
- Subir um HDFS de um nó, usar `hdfs dfs` e ler/escrever nele com Spark e Pandas
- Entender o papel do YARN e o caminho do Hadoop para a nuvem

---
## 1. Teoria: de onde veio o Hadoop

| Ano | Marco |
|-----|-------|
| 2003 | Google publica o artigo do **GFS** (Google File System) |
| 2004 | Google publica o artigo do **MapReduce** |
| 2006 | Doug Cutting e Mike Cafarella criam o **Hadoop**, implementação open source das duas ideias (o Yahoo adota em seguida) |
| 2012 | Hadoop 2 separa o gerenciamento de recursos no **YARN** |
| 2014 | **Spark** vira projeto principal da Apache: o mesmo modelo distribuído, mas em memória |
| Hoje | Na nuvem, HDFS dá lugar a object storage (S3, GCS, ADLS) e YARN a Kubernetes; on-premise, Hadoop segue em produção em muitas empresas |

A ideia central do Hadoop: **em vez de comprar um servidor gigante, use centenas de máquinas comuns** e aceite que algumas vão falhar o tempo todo. O software cuida da distribuição e das falhas.

### Os três pilares

```
  ┌────────────────────────────────────────────────────────────────────────┐
  │  PROCESSAMENTO     MapReduce   |   Spark, Hive, Flink...  (rodam aqui)  │
  ├────────────────────────────────────────────────────────────────────────┤
  │  RECURSOS          YARN: quem roda o quê, em qual máquina, com quanta   │
  │                    CPU e memória                                        │
  ├────────────────────────────────────────────────────────────────────────┤
  │  ARMAZENAMENTO     HDFS: arquivos enormes divididos em blocos e         │
  │                    replicados entre as máquinas                         │
  └────────────────────────────────────────────────────────────────────────┘
       máquina 1        máquina 2        máquina 3   ...   máquina N
```

| Pilar | Papel | Equivalente moderno mais comum |
|-------|-------|-------------------------------|
| **HDFS** | Sistema de arquivos distribuído | S3, GCS, ADLS (object storage) |
| **YARN** | Gerenciador de recursos do cluster | Kubernetes, gerenciadores de Databricks/EMR |
| **MapReduce** | Modelo e motor de processamento | Spark, Flink |

### Ecossistema

Em volta desses três pilares surgiram dezenas de projetos. Alguns que você ainda vai encontrar:

| Projeto | O que é | Situação hoje |
|---------|---------|---------------|
| **Hive** | SQL sobre arquivos no HDFS | O **Hive Metastore** ainda é usado como catálogo de tabelas por Spark, Trino etc. |
| **HBase** | Banco NoSQL sobre HDFS | Usado em nichos de baixa latência |
| **Sqoop** | Importação de bancos relacionais | Aposentado (Apache Attic) |
| **Oozie** | Orquestração de jobs | Substituído por Airflow e similares |
| **Parquet / ORC** | Formatos colunares | **Mais usados do que nunca** |

---
## 2. Setup

As seções 2 a 5 rodam em qualquer ambiente com PySpark. A seção 6 instala o Hadoop e sobe um HDFS de verdade (testado para Colab/Linux).

In [ ]:
# !pip install -q pyspark==4.0.1 pyarrow

In [ ]:
import os, re, subprocess, shutil, time, random, glob
from collections import defaultdict
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("Aula 9 - Hadoop")
    .master("local[*]")
    # spark.hadoop.* vai direto para a configuração do Hadoop. Aqui: blocos de 4 MB no disco local
    # (o padrão é 32 MB), para vermos vários blocos com um arquivo pequeno (seção 3)
    .config("spark.hadoop.fs.local.block.size", str(4 * 2**20))
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
sc = spark.sparkContext

BASE = "/tmp/aula9"
os.makedirs(BASE, exist_ok=True)

def sh(cmd, check=True):
    """Roda um comando de shell e mostra a saída (equivalente a ! no notebook)."""
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    out = (r.stdout + r.stderr).strip()
    if out:
        print(out)
    if check and r.returncode != 0:
        raise RuntimeError(f"comando falhou ({r.returncode}): {cmd}")

print("Spark            :", spark.version)
print("Hadoop (cliente) :", spark._jvm.org.apache.hadoop.util.VersionInfo.getVersion())

Repare na segunda linha: **o PySpark já vem com as bibliotecas cliente do Hadoop** dentro dele. Todo `spark.read` e `df.write` passa por elas.

---
## 3. HDFS: o sistema de arquivos distribuído

### Blocos e replicação

O HDFS divide cada arquivo em **blocos grandes** (128 MB por padrão) e guarda cada bloco em **3 máquinas diferentes**.

```
  arquivo vendas.csv (300 MB)  ->  bloco A (128 MB) | bloco B (128 MB) | bloco C (44 MB)

               NameNode (metadados: "vendas.csv = A, B, C; A está em 1, 2 e 4...")
                  │
      ┌───────────┼─────────────┬──────────────┬─────────────┐
   DataNode 1   DataNode 2   DataNode 3    DataNode 4    DataNode 5
   [A] [C]      [A] [B]      [B] [C]       [A] [C]       [B]

   Se o DataNode 2 morrer, A e B continuam disponíveis em outras máquinas,
   e o NameNode manda recriar as cópias que faltam.
```

| Componente | Função | Guarda |
|------------|--------|--------|
| **NameNode** | "Índice" do sistema de arquivos | Árvore de diretórios e onde está cada bloco, **na memória** |
| **DataNode** | Armazena os blocos | Blocos como arquivos comuns no disco local |
| **Cliente** | Pede ao NameNode onde estão os blocos e lê direto dos DataNodes | — |

### Por que blocos tão grandes

- Menos blocos significa menos metadados na memória do NameNode.
- Leitura sequencial de 128 MB amortiza o custo de buscar o bloco.
- Cada bloco é uma **unidade de paralelismo**: uma task processa um bloco.

### Limitações (e por que isso importa até hoje)

| Limitação | Consequência |
|-----------|--------------|
| Arquivos são **write-once** (só permitem append) | Não há UPDATE; daí a necessidade de Delta/Iceberg (aula 5) |
| NameNode guarda tudo em memória (~150 bytes por arquivo/bloco) | O **problema dos arquivos pequenos**: milhões de arquivos de KB derrubam o NameNode |
| Armazenamento e processamento nas mesmas máquinas | Para ter mais disco, compra-se mais CPU junto (e vice-versa) |

### Localidade de dados: "mover o código até os dados"

O princípio mais importante do Hadoop. Transferir 1 TB pela rede é lento; transferir um programa de 10 KB é instantâneo. Então o agendador tenta rodar cada task **na máquina que já tem o bloco**.

```
   Sem localidade                          Com localidade
   DataNode 1 ──[bloco A]──rede──▶ Task    DataNode 1 [bloco A] ◀── Task roda aqui
```

O Spark herdou isso: na aba Stages da Spark UI, a coluna **Locality Level** mostra `NODE_LOCAL`, `RACK_LOCAL` ou `ANY`.

### 3.1 A API de FileSystem do Hadoop

O Spark nunca fala diretamente com o disco, com o HDFS ou com o S3. Ele fala com a **interface `FileSystem` do Hadoop**, e o **esquema da URL** escolhe a implementação.

```
   spark.read.parquet("<esquema>://caminho")
                  │
     Hadoop FileSystem API (mesma interface para todos)
                  │
     ┌────────────┼──────────────┬─────────────────┬────────────────┐
   file://      hdfs://        s3a://            gs://            abfs://
   disco local  HDFS           Amazon S3         Google Cloud     Azure Data Lake
                               (hadoop-aws)      (gcs-connector)  (hadoop-azure)
```

Podemos acessar essa API diretamente pelo gateway Java do PySpark:

In [ ]:
jvm  = spark._jvm
conf = spark._jsc.hadoopConfiguration()

for scheme in ["file", "hdfs", "webhdfs", "s3a", "gs", "abfs"]:
    try:
        impl = jvm.org.apache.hadoop.fs.FileSystem.getFileSystemClass(scheme, conf).getName()
        print(f"{scheme + '://':<11} -> {impl}")
    except Exception:
        print(f"{scheme + '://':<11} -> (conector não instalado)")

`hdfs://` já funciona porque o cliente HDFS vem no PySpark. Para `s3a://`, `gs://` e `abfs://` é preciso adicionar o conector correspondente (por exemplo, o pacote `hadoop-aws` para S3). **É assim que o Spark lê de S3 até hoje: pelo conector S3A do projeto Hadoop.**

Vamos criar um arquivo de texto e inspecioná-lo com a mesma API que o Spark usa internamente.

In [ ]:
VOCAB = ("dados spark hadoop hdfs yarn mapreduce cluster bloco arquivo tabela python pandas "
         "parquet delta streaming janela modelo pipeline shuffle particao memoria disco rede "
         "driver executor task stage job lake bronze silver gold").split()
weights = [1 / (i + 1) for i in range(len(VOCAB))]          # distribuição de Zipf: poucas palavras muito frequentes

rng = random.Random(42)
CORPUS = f"{BASE}/corpus.txt"
with open(CORPUS, "w") as f:
    for _ in range(600_000):
        f.write(" ".join(rng.choices(VOCAB, weights, k=12)) + "\n")

path   = jvm.org.apache.hadoop.fs.Path(f"file://{CORPUS}")
fs     = path.getFileSystem(conf)
status = fs.getFileStatus(path)

print(f"Tamanho          : {status.getLen() / 1e6:.1f} MB")
print(f"Tamanho de bloco : {status.getBlockSize() / 2**20:.0f} MB  (definido no setup)")
print(f"Replicação       : {status.getReplication()}")
print(f"Blocos lógicos   : {-(-status.getLen() // status.getBlockSize())}")

### 3.2 Blocos e partições do Spark

Na API antiga baseada em RDD (`sc.textFile`), o Spark usa o `InputFormat` do Hadoop, e **cada bloco vira uma partição**. Na API de DataFrames (`spark.read`), o Spark calcula os pedaços por conta própria (`spark.sql.files.maxPartitionBytes`, aula 4) e usa os blocos para decidir **onde** rodar cada task.

Com os blocos de 4 MB definidos no setup:

In [ ]:
size_mb = os.path.getsize(CORPUS) / 2**20
print(f"Arquivo de {size_mb:.0f} MB com blocos de 4 MB -> {-(-size_mb // 4):.0f} blocos")
print("sc.textFile  (InputFormat do Hadoop):", sc.textFile(CORPUS).getNumPartitions(), "partições")
print("spark.read   (DataFrame)            :", spark.read.text(CORPUS).rdd.getNumPartitions(), "partições")

Lembre do prefixo **`spark.hadoop.`** usado no setup: qualquer configuração do Hadoop pode ser passada ao Spark assim. É como se configura acesso a S3 (`spark.hadoop.fs.s3a.access.key`, `spark.hadoop.fs.s3a.endpoint`...).

---
## 4. MapReduce

### O modelo

Todo processamento é expresso em duas funções que você escreve, mais uma etapa que o framework faz:

```
  ENTRADA          MAP                 SHUFFLE & SORT              REDUCE            SAÍDA
  (blocos)   (por linha: emite      (framework: agrupa       (por chave: combina
              pares chave/valor)     todos os valores         os valores)
                                     de cada chave)

  "spark hdfs"  -> (spark,1)(hdfs,1)                     hdfs  -> [1,1]  -> (hdfs, 2)
  "hdfs yarn"   -> (hdfs,1)(yarn,1)   ─── rede/disco ──▶ spark -> [1,1]  -> (spark, 2)
  "spark"       -> (spark,1)                             yarn  -> [1]    -> (yarn, 1)
```

| Etapa | Quem escreve | Paralelismo |
|-------|-------------|-------------|
| Map | Você | Uma task por bloco de entrada |
| Shuffle & Sort | O framework | Envia cada chave para um reducer (`hash(chave) % R`) |
| Reduce | Você | R tasks, cada uma recebe um conjunto de chaves **ordenadas** |

É exatamente o que acontece em um `groupBy` do Spark: o map é o stage antes do `Exchange` e o reduce é o stage depois (aula 4).

### 4.1 Simulação passo a passo

Antes de rodar de verdade, uma simulação em Python puro com 3 linhas, 2 mappers e 2 reducers, para ver cada etapa:

In [ ]:
lines = ["spark hdfs spark", "hdfs yarn", "spark yarn yarn"]
splits = [lines[:2], lines[2:]]                 # 2 "blocos" -> 2 mappers
N_REDUCERS = 2

def map_fn(line):
    for word in line.split():
        yield word, 1

def reduce_fn(word, counts):
    return word, sum(counts)

# MAP: cada mapper processa o seu bloco
map_out = [[kv for line in split for kv in map_fn(line)] for split in splits]
for i, out in enumerate(map_out):
    print(f"mapper {i}:", out)

# SHUFFLE: cada par vai para o reducer hash(chave) % R; depois, ordena e agrupa por chave
reducer_in = [defaultdict(list) for _ in range(N_REDUCERS)]
for out in map_out:
    for word, n in out:
        reducer_in[sum(map(ord, word)) % N_REDUCERS][word].append(n)   # hash simples e determinístico

print()
for r, groups in enumerate(reducer_in):
    print(f"reducer {r} recebe:", dict(sorted(groups.items())))

# REDUCE
print()
for r, groups in enumerate(reducer_in):
    print(f"reducer {r} emite :", [reduce_fn(w, c) for w, c in sorted(groups.items())])

### 4.2 Hadoop Streaming: MapReduce com scripts Python

O Hadoop é escrito em Java, mas o **Hadoop Streaming** permite usar qualquer programa que leia da entrada padrão e escreva na saída padrão. O contrato é simples:

- O **mapper** lê linhas e imprime `chave<TAB>valor`.
- O framework **ordena** tudo por chave.
- O **reducer** lê as linhas já ordenadas, então todas as ocorrências de uma chave chegam juntas.

Isso significa que dá para testar um job MapReduce inteiro no terminal, sem cluster:

```
cat entrada | python mapper.py | sort | python reducer.py
              └── map ───────┘  └shuffle┘ └── reduce ───┘
```

In [ ]:
MAPPER = r'''
import sys, re
for line in sys.stdin:
    for word in re.findall(r"[a-z]+", line.lower()):
        print(f"{word}\t1")
'''

REDUCER = r'''
import sys
current, count = None, 0
for line in sys.stdin:
    word, n = line.rstrip("\n").split("\t")
    if word != current:              # chave nova: a anterior terminou (a entrada está ordenada)
        if current is not None:
            print(f"{current}\t{count}")
        current, count = word, 0
    count += int(n)
if current is not None:
    print(f"{current}\t{count}")
'''

with open(f"{BASE}/mapper.py", "w") as f:  f.write(MAPPER)
with open(f"{BASE}/reducer.py", "w") as f: f.write(REDUCER)

print("Saída do mapper (início):")
sh(f"head -n 1 {CORPUS} | python3 {BASE}/mapper.py")

In [ ]:
t0 = time.perf_counter()
sh(f"cat {CORPUS} | python3 {BASE}/mapper.py | LC_ALL=C sort | python3 {BASE}/reducer.py > {BASE}/wordcount_mr.tsv")
t_mr = time.perf_counter() - t0

print(f"MapReduce (pipeline local): {t_mr:.1f} s\n")
mr = pd.read_csv(f"{BASE}/wordcount_mr.tsv", sep="\t", names=["word", "count"])
mr.sort_values("count", ascending=False).head(8)

> O pipeline local é lento porque roda tudo em **um processo por etapa**, e o `sort` precisa ler toda a saída do mapper antes de liberar o reducer. Em um cluster Hadoop, centenas de mappers e reducers rodam em paralelo. Mesmo assim, cada job grava a saída no HDFS, e isso pesa (seção 4.4).

### 4.3 O mesmo word count no Spark

Três versões: RDD (a API original, que imita map/reduce), DataFrame e SQL.

In [ ]:
t0 = time.perf_counter()
rdd_counts = (
    sc.textFile(CORPUS)
    .flatMap(lambda line: re.findall(r"[a-z]+", line.lower()))   # map: linha -> palavras
    .map(lambda w: (w, 1))                                       # map: palavra -> (palavra, 1)
    .reduceByKey(lambda a, b: a + b)                             # shuffle + reduce
    .collect()
)
t_rdd = time.perf_counter() - t0

t0 = time.perf_counter()
df_counts = (
    spark.read.text(CORPUS)
    .select(F.explode(F.split(F.lower("value"), " ")).alias("word"))
    .groupBy("word").count()
    .toPandas()
)
t_df = time.perf_counter() - t0

print(f"Spark RDD       : {t_rdd:.1f} s")
print(f"Spark DataFrame : {t_df:.1f} s")
print(f"MapReduce local : {t_mr:.1f} s")

check = mr.merge(df_counts, on="word", suffixes=("_mr", "_spark"))
print("\nMesmo resultado:", (check["count_mr"] == check["count_spark"]).all() and len(check) == len(mr))

### 4.4 Por que o Spark substituiu o MapReduce

O word count é um único passo. Algoritmos reais (ETL com vários joins, machine learning iterativo, grafos) são **cadeias de passos**. No MapReduce, cada passo é um job separado que **lê do HDFS e escreve no HDFS**:

```
  MapReduce
  HDFS ─▶ [map ─ shuffle ─ reduce] ─▶ HDFS ─▶ [map ─ shuffle ─ reduce] ─▶ HDFS ─▶ ...
          job 1 (disco, 3 réplicas)          job 2 (disco, 3 réplicas)

  Spark
  HDFS ─▶ [stage 1] ─shuffle─▶ [stage 2] ─shuffle─▶ [stage 3] ─▶ HDFS
          um único job, plano otimizado, dados intermediários em memória quando possível
```

| | MapReduce | Spark |
|-|-----------|-------|
| Entre etapas | Escreve no HDFS (disco + replicação) | Memória / disco local |
| Algoritmos iterativos (ML) | Um job por iteração | Dados em cache entre iterações (aula 4) |
| API | map e reduce, só | DataFrames, SQL, streaming, ML |
| Otimizador | Nenhum | Catalyst + AQE |
| Linguagens | Java (Python via Streaming) | Python, Scala, Java, R, SQL |
| Latência mínima de um job | Dezenas de segundos | Sub-segundo |

O Spark manteve as boas ideias (dividir em partições, mover o código até os dados, tolerar falhas recalculando) e eliminou o gargalo do disco.

---
## 5. YARN: quem decide onde cada coisa roda

Em um cluster compartilhado, vários times rodam jobs ao mesmo tempo. O YARN distribui CPU e memória entre eles.

```
                    ResourceManager (1 por cluster)
                    "tenho 400 núcleos e 2 TB livres"
                         │
      ┌──────────────────┼──────────────────┐
  NodeManager 1     NodeManager 2      NodeManager 3     (1 por máquina)
  ┌───────────┐     ┌───────────┐      ┌───────────┐
  │ container │     │ container │      │ container │
  │ AppMaster │     │ executor  │      │ executor  │
  │ (driver)  │     │ executor  │      │           │
  └───────────┘     └───────────┘      └───────────┘
```

| YARN | Spark |
|------|-------|
| ResourceManager | Equivale ao "master" do cluster |
| ApplicationMaster | Negocia recursos para **uma** aplicação; no modo `cluster`, o driver roda dentro dele |
| Container | Uma fatia de CPU + memória; cada **executor** do Spark é um container |
| Filas (queues) | Limitam quanto cada time pode usar |

Do lado do Spark, a única mudança é o `--master`. O mesmo código que rodamos com `local[*]` vai para o cluster assim:

```bash
spark-submit \
  --master yarn \
  --deploy-mode cluster \            # driver dentro do cluster (client = driver na sua máquina)
  --num-executors 10 \
  --executor-cores 4 \
  --executor-memory 8g \
  --queue analytics \
  pipeline_silver.py
```

| `--master` | Onde roda |
|------------|-----------|
| `local[*]` | Sua máquina (todas as aulas até aqui) |
| `yarn` | Cluster Hadoop |
| `k8s://https://...` | Kubernetes |
| `spark://host:7077` | Cluster standalone do próprio Spark |

---
## 7. Hadoop hoje: o que ficou e o que mudou

### Separação entre armazenamento e processamento

A maior mudança dos últimos 10 anos. No Hadoop clássico, as mesmas máquinas guardam os dados e processam. Na nuvem, o armazenamento é um serviço separado e barato, e o processamento sobe e desce sob demanda.

```
  Hadoop clássico (on-premise)                 Lakehouse na nuvem
  ┌──────────┐ ┌──────────┐ ┌──────────┐       ┌────────┐ ┌────────┐   clusters Spark
  │ CPU+RAM  │ │ CPU+RAM  │ │ CPU+RAM  │       │ Spark  │ │ Spark  │   sobem e descem
  │ HDFS     │ │ HDFS     │ │ HDFS     │       │ ETL    │ │ ML     │   por job
  └──────────┘ └──────────┘ └──────────┘       └───┬────┘ └───┬────┘
   cluster ligado 24h, dimensionado                 └────┬─────┘
   para o pico de disco E de CPU                ┌───────▼────────────────────┐
                                                │ S3 / GCS / ADLS             │
                                                │ Parquet + Delta / Iceberg   │
                                                └─────────────────────────────┘
```

| Componente | Situação |
|------------|----------|
| **HDFS** | Ainda forte on-premise; na nuvem, substituído por object storage |
| **MapReduce** | Praticamente só em sistemas legados |
| **YARN** | Ainda muito usado (inclusive no AWS EMR); Kubernetes cresce rápido |
| **API FileSystem** | **Onipresente**: é como Spark, Hive, Trino e Flink acessam S3/GCS/ADLS |
| **Hive Metastore** | Ainda é um catálogo de tabelas comum, convivendo com Unity Catalog, Glue, Polaris |
| **Parquet/ORC** | O padrão de armazenamento analítico |
| **Ideias** (blocos, localidade, tolerância a falhas por recomputação) | Estão em tudo, inclusive nas ferramentas da aula 8 |

### Por que ainda vale aprender Hadoop

- Muitas empresas grandes (bancos, telecom, governo, varejo) ainda rodam clusters Hadoop on-premise, e migrações levam anos.
- Erros do Spark em produção frequentemente vêm da camada Hadoop: `FileNotFoundException`, commits de saída, configuração de `s3a`, permissões no HDFS.
- Entender blocos, réplicas e localidade explica muito da performance do Spark.

---
## 9. Exercícios

In [ ]:
# ── Exercício 1 ─────────────────────────────────────────────────────────────
# MapReduce com scripts: calcule a receita total (price * quantity) por região a partir do CSV abaixo,
# escrevendo um mapper e um reducer no estilo Hadoop Streaming e rodando com
#   cat | mapper | sort | reducer
# Confira o resultado com Spark ou Pandas.
# Dica: o mapper deve pular o cabeçalho e emitir "regiao<TAB>receita".

ORDERS_CSV = f"{BASE}/orders.csv"
rng = random.Random(7)
with open(ORDERS_CSV, "w") as f:
    f.write("order_id,region,price,quantity\n")
    for i in range(200_000):
        f.write(f"{i},{rng.choice(['North','Northeast','Midwest','Southeast','South'])},"
                f"{rng.uniform(10, 500):.2f},{rng.randint(1, 5)}\n")

# Sua solução:

In [ ]:
# ── Exercício 2 ─────────────────────────────────────────────────────────────
# Dimensionamento de HDFS. Use blocos de 128 MB, replicação 3 e ~150 bytes de memória
# no NameNode por objeto (cada arquivo e cada bloco é um objeto).
#   a) Quantos blocos e quanto espaço bruto em disco ocupam 10 TB de dados?
#   b) Quanta memória do NameNode esses 10 TB consomem se estiverem em arquivos de 1 GB?
#   c) E se os mesmos 10 TB estiverem em arquivos de 100 KB?
# Relacione a resposta (c) com o problema de arquivos pequenos das aulas 4 e 5.

# Sua solução:

**Exercício 4 (discussão).** Uma empresa tem um cluster Hadoop on-premise de 40 máquinas, com 2 PB no HDFS, jobs em Hive e alguns MapReduce antigos. Ela quer migrar para a nuvem.

1. Quais partes da stack precisam mudar e quais podem ficar quase iguais?
2. Os jobs Spark existentes precisam ser reescritos? O que muda neles?
3. Que problema de custo a separação entre armazenamento e processamento resolve?

### Soluções

In [ ]:
# ── Solução 1 ───────────────────────────────────────────────────────────────
MAPPER_REV = r'''
import sys
next(sys.stdin)                                   # pula o cabeçalho
for line in sys.stdin:
    _, region, price, qty = line.rstrip("\n").split(",")
    print(f"{region}\t{float(price) * int(qty)}")
'''
REDUCER_REV = r'''
import sys
current, total = None, 0.0
for line in sys.stdin:
    region, value = line.rstrip("\n").split("\t")
    if region != current:
        if current is not None:
            print(f"{current}\t{total:.2f}")
        current, total = region, 0.0
    total += float(value)
if current is not None:
    print(f"{current}\t{total:.2f}")
'''
with open(f"{BASE}/mapper_rev.py", "w") as f:  f.write(MAPPER_REV)
with open(f"{BASE}/reducer_rev.py", "w") as f: f.write(REDUCER_REV)

sh(f"cat {ORDERS_CSV} | python3 {BASE}/mapper_rev.py | LC_ALL=C sort | python3 {BASE}/reducer_rev.py > {BASE}/rev_mr.tsv")
mr_rev = pd.read_csv(f"{BASE}/rev_mr.tsv", sep="\t", names=["region", "revenue"])

sp_rev = (spark.read.csv(ORDERS_CSV, header=True, inferSchema=True)
          .groupBy("region").agg(F.round(F.sum(F.col("price") * F.col("quantity")), 2).alias("revenue"))
          .toPandas())

check = mr_rev.merge(sp_rev, on="region", suffixes=("_mr", "_spark"))
check["diff"] = (check["revenue_mr"] - check["revenue_spark"]).abs().round(2)
check

In [ ]:
# ── Solução 2 ───────────────────────────────────────────────────────────────
TB, GB, KB, MB = 2**40, 2**30, 2**10, 2**20
BLOCK, REPL, BYTES_PER_OBJ = 128 * MB, 3, 150
data = 10 * TB

blocks = data // BLOCK
print(f"a) {blocks:,} blocos | espaço bruto = {data * REPL / TB:.0f} TB")

def namenode_memory(file_size):
    n_files  = data // file_size
    n_blocks = n_files * max(1, -(-file_size // BLOCK))      # todo arquivo ocupa pelo menos 1 bloco
    return n_files, (n_files + n_blocks) * BYTES_PER_OBJ

mems = {}
for label, size in [("b) arquivos de 1 GB  ", GB), ("c) arquivos de 100 KB", 100 * KB)]:
    n, mems[label] = namenode_memory(size)
    print(f"{label}: {n:>12,} arquivos -> {mems[label] / GB:8.2f} GB de memória no NameNode")
print(f"\nMesmo volume de dados, {max(mems.values()) / min(mems.values()):,.0f}x mais memória.")

# Mesmo volume, muito mais memória. Por isso compactamos arquivos pequenos
# (coalesce antes de escrever, aula 4; OPTIMIZE no Delta, aula 5). No S3 não há NameNode,
# mas o custo aparece de outra forma: milhões de requisições de listagem e leitura.

**Solução 4**

1. **Muda:** armazenamento (HDFS para S3/GCS/ADLS), gerenciamento de recursos (YARN para EMR/Dataproc/Databricks/Kubernetes) e o catálogo (Hive Metastore para Glue/Unity Catalog, ou um Metastore gerenciado). **Fica quase igual:** formatos (Parquet/ORC), código Spark e SQL do Hive (que roda no Spark SQL com poucos ajustes). Os jobs MapReduce são os candidatos naturais a reescrever em Spark.
2. Em geral não. Mudam os **caminhos** (`hdfs://` para `s3a://` ou `gs://`), as **configurações** (credenciais, conectores, committers de saída) e o `--master`. A lógica dos DataFrames continua a mesma, graças à API de FileSystem do Hadoop.
3. No Hadoop clássico, para ter mais disco é preciso comprar mais máquinas (com CPU e RAM juntas), e o cluster fica ligado 24 horas, dimensionado para o pico. Na nuvem, paga-se armazenamento barato o tempo todo e computação **só enquanto os jobs rodam**, com tamanho diferente para cada job.

In [ ]:
spark.stop()